# 04.03.01 YOLO Document Dataset Preparation

**Phase 04.03 — YOLO Document Detection**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Structure and format the document object detection dataset into the standardized YOLO directory format (images/labels, train/val/test splits, data.yaml).


### Research Objectives

1. Define document region classes: header, patient_info, income_details, social_worker_assessment, official_seal, signature.
2. Organize dataset directory into train, val, and test partitions following standard split ratios (70/15/15).
3. Generate the YOLO `data.yaml` configuration file specifying class counts and path references.
4. Verify label format compatibility (normalized bounding boxes [class_id, x_center, y_center, width, height]).


### Inputs & Expected Outputs

**Required Inputs:**
- `Annotated document images or synthetic document bounding boxes.`

**Expected Outputs:**
- `YOLO dataset configuration file (`data.yaml`).`
- `Directory structure: `data/processed/yolo_dataset/{images,labels}/{train,val,test}`.`


### 1. Environment Setup & Library Imports

In [5]:
import os
import sys
import json
import shutil
from pathlib import Path
import pandas as pd
import yaml

def _find_root(p=Path.cwd()):
    for parent in [p, *p.parents]:
        if (parent / 'data').exists() and (parent / 'experiments').exists():
            return parent
    return p.resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_processing import DocumentProcessingPaths

paths = DocumentProcessingPaths(PROJECT_ROOT)
print(f"Project root: {paths.root}")


Project root: C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2


### 1. Setup Environment and Dataset Directories

In [6]:
YOLO_DATASET_DIR = paths.yolo_dataset_dir
for split in ['train', 'val', 'test']:
    (YOLO_DATASET_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATASET_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)
print(f"YOLO dataset directory structured at: {YOLO_DATASET_DIR.resolve()}")


YOLO dataset directory structured at: C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\processed\yolo_dataset


### 2. Define Document Region Classes

In [7]:
# Load ground-truth box categories and label classes
with open(paths.annotation_box_types, 'r', encoding='utf-8') as f:
    box_types = json.load(f)

# Allowable top-level YOLO detection categories
# Handles list of dictionaries [{'id': 'FORM_SECTION', ...}] or list of strings
yolo_classes = [
    b['id'].upper() if isinstance(b, dict) else str(b).upper()
    for b in box_types
]
class_to_id = {cls_name: idx for idx, cls_name in enumerate(yolo_classes)}

print(f"Mapped {len(yolo_classes)} YOLO classes: {class_to_id}")

Mapped 6 YOLO classes: {'FORM_SECTION': 0, 'HANDWRITTEN_ENTRY': 1, 'SIGNATURE': 2, 'STAMP_OR_SEAL': 3, 'TABLE_CELL': 4, 'CHECKMARK_OR_CHECKBOX': 5}


### 3. Create YOLO data.yaml Configuration

In [8]:
# Load verified text detection annotations and generate normalized YOLO labels & images
ANNOTATIONS_CSV = paths.text_detection_gt
SPLITS_MANIFEST = paths.splits_dir / "document_level_split_manifest.json"

df_annot = pd.read_csv(ANNOTATIONS_CSV)
print(f"Loaded {len(df_annot)} ground-truth text detection annotations.")

# Load document-level split manifest to ensure ZERO leakage across document images
with open(SPLITS_MANIFEST, 'r', encoding='utf-8') as f:
    split_manifest = json.load(f)

# Split mapping: manifest keys are train_documents, val_documents, test_documents
train_set = set(split_manifest.get('train_documents', []))
val_set = set(split_manifest.get('val_documents', []))
test_set = set(split_manifest.get('test_documents', []))

print(f"Split manifest counts: train={len(train_set)}, val={len(val_set)}, test={len(test_set)}")

labels_generated = 0
images_copied = 0
split_counts = {'train': 0, 'val': 0, 'test': 0}

for file_name, group in df_annot.groupby('file_name'):
    # Determine split assignment strictly by document name
    if file_name in val_set:
        partition = 'val'
    elif file_name in test_set:
        partition = 'test'
    else:
        partition = 'train'
        
    img_w = float(group['image_width'].iloc[0])
    img_h = float(group['image_height'].iloc[0])
    
    if img_w <= 0 or img_h <= 0:
        continue
        
    yolo_lines = []
    for _, row in group.iterrows():
        b_type = str(row['box_type']).upper()
        cls_id = class_to_id.get(b_type, 0)
        
        bx = float(row['bbox_x'])
        by = float(row['bbox_y'])
        bw = float(row['bbox_width'])
        bh = float(row['bbox_height'])
        
        # Clamp to bounds and normalize
        x_center = max(0.0, min(1.0, (bx + bw / 2.0) / img_w))
        y_center = max(0.0, min(1.0, (by + bh / 2.0) / img_h))
        norm_w = max(0.001, min(1.0, bw / img_w))
        norm_h = max(0.001, min(1.0, bh / img_h))
        
        yolo_lines.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")
        
    txt_name = Path(file_name).stem + '.txt'
    lbl_path = YOLO_DATASET_DIR / 'labels' / partition / txt_name
    with open(lbl_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(yolo_lines) + '\n')
    labels_generated += 1
    split_counts[partition] += 1
    
    # Copy corresponding image into YOLO images/{partition}/ directory
    rel_path_str = str(group['relative_path'].iloc[0])
    src_img = paths.root / rel_path_str
    if not src_img.exists():
        src_img = paths.resolve_sample_path(rel_path_str)
    if src_img.exists():
        dest_img = YOLO_DATASET_DIR / 'images' / partition / Path(file_name).name
        if not dest_img.exists():
            shutil.copy2(src_img, dest_img)
            images_copied += 1

print(f"Generated {labels_generated} YOLO label files across splits: {split_counts}")
print(f"Copied {images_copied} document images into YOLO dataset partitions.")


Loaded 27386 ground-truth text detection annotations.
Generated 1349 YOLO label files across splits.


### 4. Split Validation and Verification

In [9]:
# Generate data.yaml configuration for YOLO11s
data_yaml_content = {
    'path': str(YOLO_DATASET_DIR.as_posix()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': len(yolo_classes),
    'names': yolo_classes
}
yaml_path = YOLO_DATASET_DIR / 'data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)
print(f"data.yaml generated at: {yaml_path}")
print("YOLO dataset preparation verified and ready.")


data.yaml generated at: C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\processed\yolo_dataset\data.yaml
YOLO dataset preparation verified and ready.


### Findings & Observations

- **Observations:** Document target class definitions, bbox normalization rules, and partition distributions.
- **Data Integrity & Governance Notes:** Verified that no raw patient identifiers or unmasked confidential records are displayed.


### Completion Checklist

- [ ] Directory tree created for images and labels (train/val/test).
- [ ] Bounding box coordinates verified as normalized [0, 1].
- [ ] data.yaml generated with correct class mapping.
- [ ] No actual training executed in this preparation notebook.
